Shared dataset: cleaning, labels, split, image cache
Owner: Ahmed

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!git clone -q https://github.com/eyadstein/cxr_second_reader.git || git -C cxr_second_reader pull
%cd /content/cxr_second_reader
!pip install -q -r requirements.txt
import sys; sys.path.append("/content/cxr_second_reader")

In [ ]:
import os, re, numpy as np, pandas as pd, kagglehub
from src.config import SHARED, INDIANA_SLUG
from src.evaluate import extract_labels
os.makedirs(SHARED, exist_ok=True)
kagglehub.login()
path = kagglehub.dataset_download(INDIANA_SLUG)
rep = pd.read_csv(f"{path}/indiana_reports.csv")
proj = pd.read_csv(f"{path}/indiana_projections.csv")
print(rep.columns.tolist()); rep.head()

In [ ]:
def clean(t):
    t = str(t) if pd.notna(t) else ""
    t = t.replace("XXXX", " ").lower()
    return re.sub(r"\s+", " ", t).strip()

for c in ["indication", "comparison", "findings", "impression"]:
    rep[c] = rep[c].map(clean)
rep = rep[(rep.findings != "") | (rep.impression != "")].drop_duplicates("uid")
rep["target_text"] = (rep.findings + ". " + rep.impression + ".").str.strip()
frontal = proj[proj.projection == "Frontal"].drop_duplicates("uid")[["uid", "filename"]]
df = rep.merge(frontal, on="uid").rename(columns={"filename": "frontal_img"})

In [ ]:
rng = np.random.default_rng(42)
uids = df.uid.unique(); rng.shuffle(uids); n = len(uids)
split = {u: "train" for u in uids[:int(.7 * n)]}
split.update({u: "val" for u in uids[int(.7 * n):int(.85 * n)]})
split.update({u: "test" for u in uids[int(.85 * n):]})
df["split"] = df.uid.map(split)
print(df.split.value_counts())

In [ ]:
# TODO labels_8: df["labels_8"] = (df.findings + " " + df.impression).map(lambda t: extract_labels(t).tolist())
# TODO image cache: save each frontal image as grayscale PNG into SHARED/img224 and SHARED/img384, named uid.png, with CLAHE. No horizontal flips anywhere.
df.to_csv(f"{SHARED}/dataset.csv", index=False)